In [0]:

df_bronce = spark.read.parquet('/Volumes/quality_data/bronze/bronce_data_quality/control_calidad.parquet')

display(df_bronce)


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
df_bronce.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("quality_data.bronze.control_calidad")

In [0]:
df_bronce.describe()

In [0]:
df_bronce.printSchema()

In [0]:
for posicion, columna in enumerate(df_bronce.columns, start=1):
    print(f"{posicion}. {columna}")

In [0]:
print(f"Total de columnas: {len(df_bronce.columns)}")

In [0]:
display(df_bronce.limit(20))

In [0]:
from pyspark.sql import functions as F

resumen_general = df_bronce.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("HDR").alias("total_hdr"),
    F.countDistinct("Cliente").alias("total_clientes"),
    F.countDistinct("Variable").alias("total_variables"),
    F.min("CCFchUti").alias("fecha_minima"),
    F.max("CCFchUti").alias("fecha_maxima")
)

display(resumen_general)

In [0]:
def es_nulo_o_vacio(nombre_columna):
    return (
        F.col(nombre_columna).isNull()
        | (F.trim(F.col(nombre_columna).cast("string")) == "")
    )

resumen_nulos = df_bronce.agg(
    F.count("*").alias("total_registros"),

    F.sum(
        F.when(es_nulo_o_vacio("HDR"), 1).otherwise(0)
    ).alias("hdr_nulos"),

    F.sum(
        F.when(F.col("Variable").isNull(), 1).otherwise(0)
    ).alias("variable_nula"),

    F.sum(
        F.when(es_nulo_o_vacio("VariableNom"), 1).otherwise(0)
    ).alias("nombre_variable_nulo"),

    F.sum(
        F.when(F.col("CCvalFloat").isNull(), 1).otherwise(0)
    ).alias("resultado_numerico_nulo"),

    F.sum(
        F.when(F.col("CCFchUti").isNull(), 1).otherwise(0)
    ).alias("fecha_nula"),

    F.sum(
        F.when(es_nulo_o_vacio("Cliente"), 1).otherwise(0)
    ).alias("cliente_nulo")
)

display(resumen_nulos)

In [0]:
total_registros = df_bronce.count()

columnas_validar = [
    "HDR",
    "Cliente",
    "Control_Calidad",
    "Variable",
    "VariableNom",
    "Resultado",
    "CCvalFloat",
    "CCFchUti",
    "ARTICULO",
    "FasCod",
    "ProCod"
]

resultado_nulos = []

for columna in columnas_validar:
    cantidad_nulos = (
        df_bronce
        .filter(es_nulo_o_vacio(columna))
        .count()
    )

    porcentaje_nulos = (
        cantidad_nulos / total_registros * 100
        if total_registros > 0
        else 0
    )

    resultado_nulos.append(
        (
            columna,
            cantidad_nulos,
            round(porcentaje_nulos, 2)
        )
    )

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["columna", "cantidad_nulos", "porcentaje_nulos"]
)

display(
    df_nulos.orderBy(F.desc("porcentaje_nulos"))
)

In [0]:
resultados_no_numericos = (
    df_bronce
    .filter(
        F.col("CCvalFloat").isNull()
        & F.col("Resultado").isNotNull()
        & (F.trim(F.col("Resultado").cast("string")) != "")
    )
    .groupBy(
        "Variable",
        "VariableNom",
        "Resultado"
    )
    .agg(
        F.count("*").alias("cantidad")
    )
    .orderBy(
        F.desc("cantidad")
    )
)

display(resultados_no_numericos.limit(100))

In [0]:
clasificacion_resultados = (
    df_bronce
    .withColumn(
        "tipo_resultado",
        F.when(
            F.col("CCvalFloat").isNotNull(),
            F.lit("NUMERICO")
        )
        .when(
            es_nulo_o_vacio("Resultado"),
            F.lit("VACIO")
        )
        .otherwise(
            F.lit("NO_NUMERICO")
        )
    )
    .groupBy("tipo_resultado")
    .agg(
        F.count("*").alias("cantidad")
    )
    .withColumn(
        "porcentaje",
        F.round(
            F.col("cantidad") / F.lit(total_registros) * 100,
            2
        )
    )
    .orderBy(F.desc("cantidad"))
)

display(clasificacion_resultados)

In [0]:
duplicados = (
    df_bronce
    .groupBy(
        "HDR",
        "Variable",
        "CCFchUti"
    )
    .agg(
        F.count("*").alias("cantidad")
    )
    .filter(
        F.col("cantidad") > 1
    )
    .orderBy(
        F.desc("cantidad")
    )
)

display(duplicados.limit(100))
total_grupos_duplicados = duplicados.count()

print(f"Grupos con posibles duplicados: {total_grupos_duplicados:,}")

In [0]:
display(
    df_bronce.select(
        "CCvalFloat",
        "Variable"
    
    ).summary()
)

In [0]:
estadisticas_variables = (
    df_bronce
    .groupBy(
        "Variable",
        "VariableNom"
    )
    .agg(
        F.count("*").alias("total_mediciones"),
        F.count("CCvalFloat").alias("mediciones_numericas"),
        F.round(
            F.avg("CCvalFloat"),
            4
        ).alias("promedio"),
        F.round(
            F.stddev("CCvalFloat"),
            4
        ).alias("desviacion_estandar"),
        F.min("CCvalFloat").alias("valor_minimo"),
        F.max("CCvalFloat").alias("valor_maximo")
    )
    .withColumn(
        "porcentaje_numerico",
        F.round(
            F.col("mediciones_numericas")
            / F.col("total_mediciones")
            * 100,
            2
        )
    )
    .orderBy(
        F.desc("total_mediciones")
    )
)

display(estadisticas_variables)

In [0]:
df_bronce.describe("CCvalFloat").show()